In [62]:
import pathlib as pl
import numpy as np
import casPYan
import caspian
import neuro
import json
import copy

In [63]:
# from casPYan import Node
# Node.int8 = False

In [64]:
# seed = 43
# seed = 983071040
seed = None
if seed is None:
    rng = np.random.default_rng()
    seed = rng.integers(0, 2**32)
print(f"Seed: {seed}")
rng = np.random.default_rng(seed)
size = 10
density = 0.2


def rif(a, b):
    return float(rng.integers(a, b, endpoint=True))

Seed: 1892740668


In [65]:
nodes = []


for _i in range(size):
    nodes.append(casPYan.Node(threshold=rif(0, 127), delay=0, leak=rif(-1, 4)))

In [66]:
weights = rng.integers(-127, 127, size=(size, size), endpoint=True)
weights

array([[ -96,  -79,   -3,  -77,   31,  -87,  -90,  -72,  119,  -29],
       [ 117,   74,   53,  -66,   88,   34,  101,   58,   43,  112],
       [  16,  -87,  -50,  -29,  -66,    1,   -5,  112,  -70,  116],
       [ -39,  -28,  -40,   62,   90,  -50,   44,  -76, -112, -101],
       [ -52,  -18, -116,  -72,  -53,   50,  -39,   -1,  -17,  -38],
       [ 125,   56,  -88,  114,  -85,   59,  -12,   97,  124,   99],
       [ -52,  -32,  -16,  -40,  -29,   32,   -9,    0,   43,  102],
       [ -13,   51,   97,   22,  -72,   67,  103,   53,   93,   43],
       [ -20,  110,    8,  -36,  -20,  -53, -117,  -20, -100,   -8],
       [ -78,  110,  -72, -112,   -8,   27,  100,   87,   64,  -93]])

In [67]:
mask = rng.choice([1, 0], size=(size, size), p=[density, 1 - density])
mask.nonzero()

(array([0, 0, 1, 1, 1, 2, 2, 3, 3, 5, 6, 7, 7, 7, 8, 8, 8, 9, 9, 9, 9]),
 array([2, 6, 1, 7, 9, 6, 8, 1, 5, 2, 0, 1, 8, 9, 0, 1, 8, 1, 2, 4, 7]))

In [68]:
for i, j in zip(*(weights * mask).nonzero()):
    casPYan.connect(nodes[i], nodes[j], weight=float(weights[i, j]), delay=int(rif(0, 255)))

In [69]:
proc = casPYan.Processor()
proc.nodes = nodes
proc.inputs = nodes[:2]
proc.outputs = nodes[-2:]

netj = proc.to_tennlab()
netj

{'Associated_Data': {'application': {},
  'label': None,
  'processor': {'Leak_Enable': True,
   'Max_Leak': 4,
   'Min_Leak': -1,
   'Max_Weight': 127,
   'Min_Weight': -127,
   'Max_Threshold': 127,
   'Min_Threshold': 0,
   'Max_Synapse_Delay': 255,
   'Min_Synapse_Delay': 0,
   'Max_Axon_Delay': 0,
   'Min_Axon_Delay': 0}},
 'Properties': {'network_properties': [],
  'node_properties': [{'index': 0,
    'name': 'Threshold',
    'max_value': 127.0,
    'min_value': 0.0,
    'size': 1,
    'type': 73},
   {'index': 1,
    'name': 'Leak',
    'max_value': 4.0,
    'min_value': -1.0,
    'size': 1,
    'type': 73},
   {'index': 2,
    'name': 'Delay',
    'max_value': 0.0,
    'min_value': 0.0,
    'size': 1,
    'type': 73}],
  'edge_properties': [{'index': 0,
    'name': 'Weight',
    'max_value': 127.0,
    'min_value': -127.0,
    'size': 1,
    'type': 73},
   {'index': 1,
    'name': 'Delay',
    'max_value': 300.0,
    'min_value': 0.0,
    'size': 1,
    'type': 73}]},
 'Nodes'

In [70]:
pydut = casPYan.Processor()
pydut.load_network(netj)

In [71]:
cnet = neuro.Network()
cnet.from_json(netj)
cpdut = caspian.Processor(cnet.get_data("processor"))
cpdut.load_network(cnet)
cpdut.track_neuron_events(True)
cpdut.get_params()

{"binary_input":true,"input_scaling_value":255,"integration_delay":true,"plasticity":"none","run_time_inclusive":false,"spike_raster_info":true,"threshold_inclusive":false}

In [72]:
cpdut.get_internal_network(0)

{
  "config": {
    "max_axon_delay": 0,
    "max_syn_delay": 255,
    "max_threshold": 255,
    "soft_reset": false
  },
  "inputs": [
    0,
    1
  ],
  "neurons": [
    {
      "delay": 0,
      "id": 0,
      "leak": 2,
      "threshold": 85
    },
    {
      "delay": 0,
      "id": 1,
      "leak": 1,
      "threshold": 18
    },
    {
      "delay": 0,
      "id": 2,
      "leak": 1,
      "threshold": 51
    },
    {
      "delay": 0,
      "id": 3,
      "leak": 3,
      "threshold": 88
    },
    {
      "delay": 0,
      "id": 4,
      "leak": 0,
      "threshold": 32
    },
    {
      "delay": 0,
      "id": 5,
      "leak": -1,
      "threshold": 120
    },
    {
      "delay": 0,
      "id": 6,
      "leak": -1,
      "threshold": 37
    },
    {
      "delay": 0,
      "id": 7,
      "leak": 3,
      "threshold": 50
    },
    {
      "delay": 0,
      "id": 8,
      "leak": 2,
      "threshold": 38
    },
    {
      "delay": 0,
      "id": 9,
      "leak": 2,
      "

In [73]:
def spike_py2c(spikes: list[list[tuple[float, int]]]):
    return [neuro.Spike(id=nid, time=t, value=v / 255) for nid, n_spikes in enumerate(spikes) for v, t in n_spikes]

In [74]:
def random_spikes(n, t=1, value_range=(0.0, 1.0), choices=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    if isinstance(n, int):
        n = range(n)
    if choices is None:
        for _i in n:
            yield [(float(v), i) for i, v in enumerate(rng.uniform(*value_range, t))]
    else:
        for _i in n:
            yield [(float(v), i) for i, v in enumerate(rng.choice(choices, t))]

In [75]:
spikes = list(random_spikes(proc.inputs, 10, choices=[0, 100], rng=rng))
print(spikes)

[[(0.0, 0), (100.0, 1), (100.0, 2), (100.0, 3), (100.0, 4), (100.0, 5), (0.0, 6), (0.0, 7), (100.0, 8), (0.0, 9)], [(0.0, 0), (0.0, 1), (0.0, 2), (100.0, 3), (100.0, 4), (0.0, 5), (100.0, 6), (100.0, 7), (0.0, 8), (100.0, 9)]]


In [76]:
pydut.apply_spikes(spikes)
pydut.run(5)
pydut.run(5)
print()
print(pydut.neuron_vectors())
print(pydut.neuron_fires())
print(pydut.neuron_charges())


[[0, 1, 4], [0, 2, 3], [], [], [], [], [], [], [], []]
[3, 3, 0, 0, 0, 0, 0, 0, 0, 0]
[0.0, 100.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [77]:
pydut.nodes[1]

Node at 7de0b2e29090 w/ Threshold: 18.0, Delay: 0, Leak: 1.0, children: ['9090', '96d0', '98b0']

In [78]:
# print([(s.id, s.time, s.value) for s in spike_py2c(spikes)])

In [79]:
cpdut.apply_spikes(spike_py2c(spikes))
cpdut.run(5)
cpdut.run(5)
print(cpdut.neuron_vectors())
print(cpdut.neuron_counts())
print(cpdut.neuron_charges())

[[0.0, 1.0, 4.0], [0.0, 2.0, 3.0], [], [], [], [], [], [], [], []]
[3, 3, 0, 0, 0, 0, 0, 0, 0, 0]
[0.0, 100.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [80]:
from scipy.sparse import coo_matrix

In [81]:
ones, row, col = np.array([
    (1, t, int(v))
    for t, vec in enumerate(cpdut.neuron_vectors())
    for v in vec]).T
s = coo_matrix((ones, (row, col))).todense()
s

matrix([[1, 1, 0, 0, 1],
        [1, 0, 1, 1, 0]])